# PTB-XL Data Pipeline

## Objective

This notebook develops and validates the data pipeline used to transform raw PTB-XL ECG recordings and diagnostic annotations into model-ready PyTorch inputs.

The reusable implementation is kept in the `src/` directory, while this notebook is used to inspect intermediate outputs and verify that each stage of the pipeline behaves correctly.

## 1. Setup

In [13]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "ptb-xl"

sys.path.append(str(PROJECT_ROOT))

from src.preprocessing import load_ecg, compute_dataset_stats, normalize_signal

In [14]:
df = pd.read_csv(DATA_DIR / "ptbxl_database.csv")

df[["ecg_id", "filename_lr"]].head()

,ecg_id,filename_lr
0,1,records100/00000/00001_lr
1,2,records100/00000/00002_lr
2,3,records100/00000/00003_lr
3,4,records100/00000/00004_lr
4,5,records100/00000/00005_lr


## 2. Loading ECG Signals

PTB-XL stores ECG recordings using the WFDB format used by PhysioNet.

The `wfdb.rdsamp()` function reads a recording and returns the ECG waveform together with metadata describing the signal.

The 100 Hz PTB-XL recordings contain 1,000 time samples across 12 ECG leads, giving an initial signal shape of `(1000, 12)`.

The signal is transposed to `(12, 1000)` so that each ECG lead can later be treated as an input channel by a PyTorch 1D convolutional model.

In [15]:
record_path = DATA_DIR / df.iloc[0]["filename_lr"]
print(record_path)

test_signal = load_ecg(str(record_path))
print(test_signal.shape)

/Users/jerrywong/ML-repos/ecg-classification/data/ptb-xl/records100/00000/00001_lr
(12, 1000)


## 3. ECG Signal Normalization

Before ECG recordings are passed to a neural network, their numerical scale should be made consistent.

For this project, each of the 12 ECG leads will be standardized independently using a mean and standard deviation calculated from the training set, thus preserving relative amplitude differences between recordings.

The same training-derived statistics will then be applied to the training, validation, and test recordings.

### 3.1 Computing Training-Set Statistics

The standardization process requires a mean and standard deviation for each of the 12 ECG leads.

These statistics are calculated using only the 17,084 ECG recordings in the training set to prevent information from the validation or test sets from influencing preprocessing.

The statistics are computed across all time points for each lead, producing one mean and one standard deviation per lead.

In [16]:
import ast

scp = pd.read_csv(DATA_DIR / "scp_statements.csv", index_col=0)

diagnostic_scp = scp[scp["diagnostic"] == 1]

df["scp_codes_parsed"] = df["scp_codes"].apply(ast.literal_eval)

In [17]:
def get_diagnostic_classes(codes):
    classes = set()

    for code in codes:
        if code in diagnostic_scp.index:
            diagnostic_class = diagnostic_scp.loc[code, "diagnostic_class"]

            if pd.notna(diagnostic_class):
                classes.add(diagnostic_class)

    return sorted(classes)

In [18]:
df["diagnostic_classes"] = df["scp_codes_parsed"].apply(
    get_diagnostic_classes
)

model_df = df[
    df["diagnostic_classes"].apply(len) > 0
].copy()

train_df = model_df[
    model_df["strat_fold"].between(1, 8)
].copy()

print(len(train_df))

17084


In [19]:
train_filepaths = [
    DATA_DIR / filename
    for filename in train_df["filename_lr"]
]

In [20]:
train_mean, train_std = compute_dataset_stats(train_filepaths)

print("Mean shape:", train_mean.shape)
print("Standard deviation shape:", train_std.shape)

print("\nPer-lead means:")
print(train_mean)

print("\nPer-lead standard deviations:")
print(train_std)

Mean shape: (12,)
Standard deviation shape: (12,)

Per-lead means:
[-0.00174681 -0.00152129  0.00022546  0.00161655 -0.00093192 -0.00062547
  0.00017763 -0.00095178 -0.00156507 -0.00135631 -0.00079977 -0.00242425]

Per-lead standard deviations:
[0.16001027 0.15922999 0.15997245 0.13814795 0.1387673  0.13797789
 0.23434961 0.33421614 0.32846893 0.29276388 0.27206945 0.28096763]


### 3.2 Applying Standardization

For each lead and time point, standardization is performed using:

$$
x' = \frac{x-\mu}{\sigma}
$$

where:

- $x$ is the original ECG value
- $\mu$ is the training-set mean for that lead
- $\sigma$ is the training-set standard deviation for that lead

The same training-derived statistics are applied to the training, validation, and test recordings.

In [21]:
normalized_signal = normalize_signal(
    test_signal,
    train_mean,
    train_std
)

print(normalized_signal.shape)
print(normalized_signal.mean(axis=1))
print(normalized_signal.std(axis=1))

(12, 1000)
[ 0.02034127  0.01409466 -0.00627268 -0.01975817  0.01544252  0.00458387
 -0.00805049  0.02378933 -0.00064521 -0.00933411  0.00089232  0.01158942]
[0.68134512 0.52266105 0.367124   0.66872304 0.55465179 0.33964123
 0.48268681 0.64099127 0.35873668 0.32590994 0.32787359 0.36281554]


## 4. Multi-Hot Label Encoding

PTB-XL is a multi-label classification problem because a single ECG recording can contain multiple diagnostic superclasses.

The diagnostic labels are converted from lists of class names into fixed-length multi-hot vectors that can be used as targets for a neural network.

The class order is fixed as:

`NORM`, `MI`, `STTC`, `CD`, `HYP`

Each position in the output vector corresponds to whether that diagnostic superclass is present.

In [28]:
model_df = df[
    df["diagnostic_classes"].apply(len) > 0
].copy()

In [29]:
train_df = model_df[
    model_df["strat_fold"].between(1, 8)
].copy()

val_df = model_df[
    model_df["strat_fold"] == 9
].copy()

test_df = model_df[
    model_df["strat_fold"] == 10
].copy()

In [30]:
CLASSES = [
    "NORM",
    "MI",
    "STTC",
    "CD",
    "HYP"
]


def encode_labels(labels):
    """
    Convert diagnostic classes into a multi-hot vector.

    Args:
        labels: List of diagnostic classes.

    Returns:
        Multi-hot encoded label vector.
    """
    return [    
        1 if cls in labels else 0
        for cls in CLASSES
    ]

In [31]:
print(encode_labels(["MI", "STTC"]))
print(encode_labels(["NORM"]))
print(encode_labels(["CD", "NORM"]))

[0, 1, 1, 0, 0]
[1, 0, 0, 0, 0]
[1, 0, 0, 1, 0]


### 4.1 Applying Label Encoding

The multi-hot encoding function is applied to each ECG in the training, validation, and test datasets.

This creates a fixed-length target vector for each recording that can be directly used as the output target during model training.

In [32]:
train_df["labels"] = train_df["diagnostic_classes"].apply(
    encode_labels
)

val_df["labels"] = val_df["diagnostic_classes"].apply(
    encode_labels
)

test_df["labels"] = test_df["diagnostic_classes"].apply(
    encode_labels
)

In [27]:
train_df[
    ["ecg_id", "diagnostic_classes", "labels"]
].head()

,ecg_id,diagnostic_classes,labels
0,1,[NORM],"[1, 0, 0, 0, 0]"
1,2,[NORM],"[1, 0, 0, 0, 0]"
2,3,[NORM],"[1, 0, 0, 0, 0]"
3,4,[NORM],"[1, 0, 0, 0, 0]"
4,5,[NORM],"[1, 0, 0, 0, 0]"
